<a href="https://colab.research.google.com/github/4GeeksAcademy/Wiston_Travieso_Tutorial-de-Clasficador-de-Imagenes/blob/main/Sistema%20de%20clasificaci%C3%B3n%20de%20im%C3%A1genes.es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Sistema de clasificación de imágenes

El conjunto de datos se compone de fotos de perros y gatos proporcionadas como un subconjunto de fotos de uno mucho más grande de 3 millones de fotos anotadas manualmente. Estos datos se obtuvieron a través de una colaboración entre Petfinder.com y Microsoft.

El conjunto de datos se usó originalmente como un CAPTCHA, es decir, una tarea que se cree que un humano encuentra trivial, pero que una máquina no puede resolver, que se usa en sitios web para distinguir entre usuarios humanos y bots. La tarea se denominó "Asirra". Cuando se presentó "Asirra", se mencionó "que los estudios de usuarios indican que los humanos pueden resolverlo el 99,6% de las veces en menos de 30 segundos". A menos que se produzca un gran avance en la visión artificial, esperamos que los ordenadores no tengan más de 1/54.000 posibilidades de resolverlo.

En el momento en que se publicó la competencia, el resultado de última generación se logró con un SVM y se describió en un artículo de 2007 con el título "Ataques de Machine Learning contra el CAPTCHA de Asirra" (PDF) que logró una precisión de clasificación del 80%. Fue este documento el que demostró que la tarea ya no era una tarea adecuada para un CAPTCHA poco después de que se propusiera la tarea.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.preprocessing import image


#### Paso 1: Carga del conjunto de datos

El conjunto de datos se encuentra en este link. Descarga la carpeta y descomprime los archivos. Ahora tendrás una carpeta con el dataset y una carpeta llamada train que contiene más de 25.000 archivos de imagen (formato .jpg) de perros y gatos. Las fotos están etiquetadas por su nombre de archivo, con la palabra dog o cat.

In [ ]:
# Importar 10 fotos de perros desde carpeta zip
data =

In [ ]:
y_pred = model.predict(X_test)
y_pred[:15]

5/5 [==============================] - 0s 783us/step


array([[2.6933843e-01],
       [5.7993677e-02],
       [7.6992743e-02],
       [4.8524177e-01],
       [3.1675667e-01],
       [6.4265609e-01],
       [7.3388085e-04],
       [2.8476545e-01],
       [8.7694836e-01],
       [4.1469648e-01],
       [1.6080230e-01],
       [8.2213795e-01],
       [2.1518065e-01],
       [5.3527528e-01],
       [1.2730679e-01]], dtype=float32)

Como vemos, el modelo no devuelve las clases `0` y `1` directamente, sino que requiere de un preprocesamiento previo:

In [ ]:
y_pred_round = [round(x[0]) for x in y_pred]
y_pred_round[:15]

[0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0]

Con los datos en crudo es muy complicado saber si el modelo está acertando o no. Para ello, debemos compararlo con la realidad. Existe una gran cantidad de métricas para medir la efectividad de un modelo a la hora de predecir, entre ellas la **precisión** (*accuracy*), que es la fracción de predicciones que el modelo realizó correctamente.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred_round)

0.7272727272727273

#### Paso 4: Guardado del modelo

Una vez tenemos el modelo que estábamos buscando (presumiblemente tras la optimización de hiperparámetros), para poder utilizarlo a futuro es necesario almacenarlo en nuestro directorio.

In [ ]:
model.save("keras_8-12-8-1_42.keras")

Añadir un nombre explicativo al modelo es vital, ya que en el caso de perder el código que lo ha generado sabremos qué arquitectura tiene (en este caso decimos `8-12-8-1` porque tiene 8 neuronas en la capa de entrada, 12 y 8 en las dos capas ocultas y una neurona en la capa de salida) y además la semilla para replicar los componentes aleatorios del modelo, que en este caso lo hacemos añadiendo un número al nombre del archivo, el `42`.

### Clasificación de conjuntos de imágenes

A continuación se muestra un ejemplo simple de cómo entrenar una red neuronal para clasificar imágenes del dataset MNIST. MNIST es un conjunto de datos de imágenes de dígitos escritos a mano, desde 0 hasta 9.

#### Paso 1. Lectura del conjunto de datos

In [ ]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalizar los datos (transformamos los valores de los píxeles de 0-255 a 0-1)
X_train, X_test = X_train / 255.0, X_test / 255.0

11490434/11490434 [==============================] - 1s 0us/step


Los valores de los píxeles de las imágenes se normalizan para que estén en el rango de 0 a 1 en lugar de 0 a 255.

#### Paso 2: Inicialización y entrenamiento del modelo

Se define la arquitectura de la red neuronal. En este caso, estamos utilizando un modelo secuencial simple con una capa de aplanamiento que transforma las imágenes 2D en vectores 1D, una capa densa con 128 neuronas y una capa de salida con 10 neuronas.

A continuación se proporciona una forma alternativa a la anterior para crear una RNA. Ambas son válidas:

In [ ]:
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import set_random_seed

set_random_seed(42)

model = Sequential([
  # Capa que aplana la imagen de entrada de 28x28 píxeles a un vector de 784 elementos
  Flatten(input_shape = (28, 28)),
  # Capa oculta densa con 128 neuronas y función de activación ReLU
  Dense(128, activation = "relu"),
  # Capa de salida con 10 neuronas (una para cada dígito del 0 al 9)
  Dense(10)
])

También añadimos el compilador de la red para definir el optimizador y la función de pérdida, como hicimos anteriormente:

In [ ]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model.compile(optimizer = "adam", loss = SparseCategoricalCrossentropy(from_logits = True), metrics = ["accuracy"])

Se entrena el modelo en el conjunto de entrenamiento durante un cierto número de épocas. Cuando se trabaja con imágenes es menos común utilizar el parámetro del `batch_size`:

In [ ]:
model.fit(X_train, y_train, epochs = 5)

Epoch 1/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.2530 - accuracy: 0.9276
Epoch 2/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.1111 - accuracy: 0.9671
Epoch 3/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.0759 - accuracy: 0.9757
Epoch 4/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.0566 - accuracy: 0.9831
Epoch 5/5
1875/1875 [==============================] - 2s 1ms/step - loss: 0.0432 - accuracy: 0.9865


In [ ]:
_, accuracy = model.evaluate(X_train, y_train)

print(f"Accuracy: {accuracy}")

1875/1875 [==============================] - 1s 671us/step - loss: 0.0441 - accuracy: 0.9858
Accuracy: 0.9858166575431824


El tiempo de entrenamiento de un modelo dependerá, en primer lugar, del tamaño del conjunto de datos (instancias y características), y también de la tipología de modelo y su configuración.

#### Paso 3: Predicción del modelo

In [ ]:
test_loss, test_acc = model.evaluate(X_test,  y_test, verbose=2)

print('\nTest accuracy:', test_acc)

313/313 - 0s - loss: 0.0841 - accuracy: 0.9751 - 266ms/epoch - 851us/step

Test accuracy: 0.9750999808311462


#### Paso 4: Guardado del modelo

Una vez tenemos el modelo que estábamos buscando (presumiblemente tras la optimización de hiperparámetros), para poder utilizarlo a futuro es necesario almacenarlo en nuestro directorio.

In [ ]:
model.save("keras_28x28-128-10_42.keras")

Añadir un nombre explicativo al modelo es vital, ya que en el caso de perder el código que lo ha generado sabremos qué arquitectura tiene (en este caso ponemos `28x28-128-10` porque tiene una capa de entrada de 28 x 28 píxeles, 128 neuronas en la única capa oculta que tiene y 10 neuronas en la capa de salida) y además la semilla para replicar los componentes aleatorios del modelo, que en este caso lo hacemos añadiendo un número al nombre del archivo, el `42`.